# KAIROS — Démonstration de reproductibilité (Acte 2)

**Rejeu d'un artefact scellé, depuis une source publique — aucun composant propriétaire Kairos.**

> ⚠️ **Ce notebook Colab est un véhicule de reproduction alternatif — ce n'est pas l'environnement de confiance de Kairos.**
> Le replay sur votre propre machine locale reste le chemin de référence, avec la plus grande valeur probatoire. Voir la section "Prérequis" et "Exécution" du dossier complet pour la procédure locale.

Run all cells (Runtime → Run all). ~2-3 minutes. Aucune installation préalable.

Ce notebook télécharge le dataset public BBBP depuis sa source d'origine, reconstruit l'artefact Parquet avec l'enveloppe exacte (Python 3.10, pandas 1.5.3, pyarrow 23.0.0), et compare l'empreinte obtenue à celle scellée le 2026-06-06.

Dossier technique complet : voir VALIDATION_AND_EVIDENCE.html

In [ ]:
#@title Step 1 — Bootstrap micromamba (fournit Python 3.10 quel que soit l'hôte)
!curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj bin/micromamba
!./bin/micromamba create -y -p ./env python=3.10.12 pip -c conda-forge > /dev/null 2>&1
print("Environnement Python 3.10 prêt.")

> ℹ️ **À l'étape suivante, pip peut afficher du texte rouge "ERROR: pip's dependency resolver..."** — c'est un conflit avec des packages préinstallés par Colab (`google-colab`, `numba`), sans rapport avec ce script. Les bonnes versions sont installées quand même. **Continuez à l'étape suivante**, ce n'est pas un échec.

In [ ]:
#@title Step 2 — Installer l'enveloppe exacte
!./env/bin/pip install -q pandas==1.5.3 pyarrow==23.0.0 "numpy<2"

In [ ]:
#@title Step 3 — Récupérer et vérifier le script de rejeu (fourni par le dossier)
script = r'''#!/usr/bin/env python3
"""
KAIROS — Demonstration de reproductibilite (Acte 2)
Rejeu d'un artefact enregistre, a partir d'une source publique.

Aucun composant proprietaire Kairos. Trois appels de bibliotheques open source.
L'empreinte produite doit correspondre a celle scellee le 2026-06-06.

Prerequis :
    pip install "pandas==1.5.3" "pyarrow==23.0.0" "numpy<2"
Usage :
    python3 kairos_replay_bbbp.py
"""
import hashlib
import urllib.request
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

SOURCE = "https://deepchemdata.s3.us-west-1.amazonaws.com/datasets/BBBP.csv"
CSV_SHA256_ATTENDU = "d07a38487aeac5cee5508413e468043ef3097451d2a112701c2d60be9ec6b662"
PARQUET_SHA256_SCELLE = "4621ac8d5d4a728a169fb4d5b8c35682b954928a019d575e3eca140cd563489f"


def sha256(path):
    return hashlib.sha256(open(path, "rb").read()).hexdigest()


def main():
    checks = []

    # 1. Telecharger le dataset depuis sa source publique
    urllib.request.urlretrieve(SOURCE, "BBBP.csv")
    csv_h = sha256("BBBP.csv")
    checks.append(("Source SHA-256", csv_h == CSV_SHA256_ATTENDU))

    # 2. Transformation : trois appels de bibliotheques open source
    df = pd.read_csv("BBBP.csv", low_memory=False)
    t = pa.Table.from_pandas(df)
    pq.write_table(t, "bbbp.parquet", compression="snappy")

    # 3. Empreinte et comparaison avec l'artefact scelle
    obtenu = sha256("bbbp.parquet")
    checks.append(("Reconstruction", len(df) == 2050))
    checks.append(("Sealed artifact", obtenu == PARQUET_SHA256_SCELLE))

    result = all(ok for _, ok in checks)

    print()
    print("KAIROS REPLAY")
    print("-" * 40)
    for label, ok in checks:
        print(f"{label:<22}{'MATCH' if ok else 'DIVERGE'}")
    print()
    print(f"{'RESULT':<22}{'PASS' if result else 'FAIL'}")
    print("-" * 40)
    print()
    if not result:
        print("DIVERGENCE — ne pas interpreter silencieusement.")
        print("Voir le dossier complet, section Lecture des ecarts.")
        print()
    print("Detail :")
    print("  parquet obtenu :", obtenu)
    print("  parquet scelle :", PARQUET_SHA256_SCELLE)
    print()
    print("Ce resultat provient d'un vehicule d'execution (Colab/Codespaces/Kaggle).")
    print("La preuve est le resultat deterministe et son empreinte, pas la plateforme utilisee.")


if __name__ == "__main__":
    main()
'''
with open("kairos_replay_bbbp.py", "w") as f:
    f.write(script)

import hashlib
got = hashlib.sha256(open("kairos_replay_bbbp.py", "rb").read()).hexdigest()
print("SHA-256 du script :", got)


In [ ]:
#@title Step 4 — Exécuter le rejeu
!./env/bin/python kairos_replay_bbbp.py